# SIMT 内存层级

## 概述

前面三节我们依次理解了 SIMT 的硬件架构、线程组织方式，以及如何编写核函数。上一节的核函数示例中，数据通过指针在 Global Memory 里读写——但 Global Memory 只是 SIMT 程序能用的存储之一。本节就来系统地回答：**一个 SIMT 程序在执行时，数据到底放在哪里？不同的存储介质各有什么特点？我们又能如何配置它们来获得更好的性能？**

理解内存层级，是写出高性能 SIMT 算子的关键一步——因为很多性能问题，本质上都是「数据放错了地方」或「空间没配置好」。

### 前置要求

- 已理解 SIMT 核函数的基本写法，以及线程索引如何决定每个线程处理的数据位置。
- 了解 Global Memory、Unified Buffer、Register 等基础硬件资源名称。
- 本小节为理论讲解，不依赖在线硬件环境。

### 学习目标

学完本小节后，你应该能够：

- 区分 SIMT 编程中的三类存储介质——全局内存、共享内存、寄存器，并说明各自的作用域、生命周期和用途。
- 说明 Unified Buffer 的空间划分：静态内存、动态内存、预留空间和 Data Cache。
- 掌握 Data Cache 的容量约束（至少 32KB、最多 128KB）及其计算公式。
- 理解**最大线程数**与**每个线程可用寄存器数**之间的关系，以及如何通过 `__launch_bounds__` / `__maxnreg__` 调整。

### 小节内容

- 三类存储介质总览
- 全局内存 Global Memory
- 共享内存与 Unified Buffer
- Unified Buffer 的空间划分与 Data Cache 约束
- 寄存器与最大线程数的关系

### 三类存储介质总览

SIMT 线程在执行时可以访问多种内存空间。最常打交道的是三类，它们的核心区别在于**谁能看到这块内存（作用域）**、**它能活多久（生命周期）**，以及**它在物理上位于哪里**：

| 存储介质 | 作用域（谁可访问） | 生命周期 | 物理位置 |
| --- | --- | --- | --- |
| 全局内存 Global Memory | 整个 Grid 的所有线程 | 应用程序 | Device |
| 共享内存（Unified Buffer 内） | 同一线程块（Block）内的线程 | 核函数 | Vector Core |
| 寄存器 Register | 单个线程私有 | 核函数 | Vector Core |

下面这张图把三者的包含关系画在一起，建议按「全局共享 → 线程块共享 → 线程私有」的顺序来理解：

![SIMT内存层级](images/simt_memory_hierarchy.svg)

一个关键规律是：**作用域越小，离计算单元越近，访问越快，但容量也越小**。

- 访问速度：寄存器 > 共享内存 > 全局内存
- 容量大小：全局内存 > 共享内存 > 寄存器

从程序设计角度看，这决定了数据该往哪放：全局输入输出放 Global Memory；线程块内需要协作共享的数据放共享内存；每个线程自己的临时变量放寄存器。接下来分别看这三类。

### 全局内存 Global Memory

全局内存是 Device 侧、整个 Grid 中**所有线程都能直接访问**的内存空间，也就是我们常说的 Global Memory。它有两个突出特点：

- **容量最大**，用来承载算子的输入输出等全局数据。
- **具有持久性**：分配的空间和数据会一直保留，直到被显式释放或应用程序结束。

它的典型使用流程是：用户在核函数启动前，通过 Runtime API 完成全局内存的分配与初始化；核函数执行期间，每个线程都可以读写全局内存；执行完毕后，再把结果从 Device 拷回 Host。

在核函数内部，运行在 Device 侧的代码可以**通过指针直接访问**全局内存。前面核函数小节里见过的 `float*`、`int32_t*` 这类指针参数，指向的就是 Global Memory。例如向量加法：

```cpp
__global__ void add_custom(float* x, float* y, float* z, uint64_t total_length)
{
    // 计算全局线程编号
    int32_t idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= total_length) {
        return;
    }
    z[idx] = x[idx] + y[idx];
}
```

这里 `x`、`y`、`z` 都指向 Global Memory，每个线程根据自己算出的 `idx` 访问不同的数据位置，互不重叠。

### 共享内存与 Unified Buffer

共享内存是**同一线程块内所有线程**都能访问的内存空间，物理上位于每个 Vector Core 内部的 **Unified Buffer**。和全局内存相比，它容量较小，但带宽更高、访问延迟更低，可以把它理解为核函数执行期间、由用户管理的高速缓存资源。线程块内的线程通过共享内存交换数据、协同完成计算。

用户可以用**静态**或**动态**两种方式申请共享内存。

**静态申请**：大小在编译期确定，不可动态修改，通过数组方式声明。

```cpp
__global__ void add_custom(...)
{
    __ubuf__ half static_buf[1024];
    // ...
}
```

默认情况下，申请到的静态内存首地址按数据类型对齐，也支持用 `__align__(N)` 手动指定对齐。

**动态申请**：大小在运行期确定。Device 侧用 `extern __ubuf__` 声明，Host 侧通过核函数调用 `<<<>>>` 配置中的第三个参数 `dyn_ubuf_size` 指定大小。

```cpp
// Device 侧：声明动态共享内存
__global__ void add_custom(...)
{
    extern __ubuf__ char dynamic_buf[];
    // ...
}

// Host 侧：通过 <<<>>> 第三个参数指定动态共享内存大小
uint32_t dyn_ubuf_size = 1024 * sizeof(char);
add_custom<<<blocks_per_grid, threads_per_block, dyn_ubuf_size, stream>>>(...);
```

无论静态还是动态，申请的共享内存都来自同一块 Unified Buffer。而 Unified Buffer 并不能全部给用户使用——这就引出了下面的空间划分问题。

### Unified Buffer 的空间划分与 Data Cache 约束

Unified Buffer 总大小为 **256KB**，但它不仅用作用户的共享内存，还要预留一部分给内部机制和 Data Cache。因此**不能把整块 Unified Buffer 都当作共享内存来申请**。

它按功能从低地址到高地址划分为四个区域：

![Unified Buffer空间划分](images/unified_buffer_layout.svg)

| 区域 | 作用 |
| --- | --- |
| 静态内存 | 用户通过静态方式申请的共享内存 |
| 动态内存 | 用户通过动态方式申请的共享内存 |
| 预留空间 | 编译器和 Ascend C 预留，**固定 8KB** |
| Data Cache | SIMT 专有的数据缓存，用于线程访问 Global Memory 时缓存数据 |

**Data Cache 的容量约束**是本节最容易踩坑的地方。Data Cache 的大小不是固定的，而是由前面几块用掉多少决定的：

```text
Data Cache 空间大小 = 256KB（UB 总大小） − 静态内存 − 动态内存 − 预留空间（8KB）
```

它有明确的上下限：

- **下限 32KB**：必须为 Data Cache 至少保留 32KB。如果用户申请的静态/动态内存太多，导致 Data Cache 不足 32KB，会出现校验报错。
- **上限 128KB**：即使静态、动态内存申请得很少，Data Cache 的实际大小也不会超过 128KB。

换句话说，配置共享内存时要量力而行：申请的静态 + 动态内存不能挤占掉 Data Cache 的 32KB 底线。

### 寄存器与最大线程数的关系

寄存器是**每个线程私有**的存储资源，访问速度最快，生命周期与核函数一致，用于保存线程的局部变量和中间结果。寄存器由编译器自动管理。

这里有一个对性能影响很大的机制：**最大线程数与每个线程可用寄存器数是相互制约的**。一个 Vector Core 的寄存器总量是固定的，参与的线程越多，平摊到每个线程的寄存器就越少。两者的对应关系如下：

**表：最大线程数与每个线程可用寄存器数**

| 最大线程数 | 每个线程可用寄存器数 |
| --- | --- |
| 1~256 | 127 |
| 257~512 | 64 |
| 513~1024 | 32 |
| 1025~2048 | 16 |

可以看到，最大线程数从 256 提到 2048，每个线程可用寄存器从 127 骤降到 16。

**为什么这会影响性能？** 如果配置的线程数过大，而每个线程的计算逻辑又比较复杂，编译器就可能没有足够的寄存器来保存局部变量，只能把一部分数据临时放到堆栈空间（Global Memory），这种现象称为**寄存器溢出（register spill）**。一旦发生溢出，原本在寄存器上的快速访问就变成了对全局内存的慢速访问，严重时甚至会"爆栈"导致精度异常、影响算子功能。

所以配置时需要权衡：

- 线程数**太大** → 每个线程寄存器太少 → 复杂计算容易触发 register spill，性能下降。
- 线程数**太小** → 实际能启动的线程数受限 → 并行度不足，浪费硬件资源。

**如何调整**：Ascend C 提供了两个可选关键字，在定义核函数时，从相对的两端来配置这一对相互制约的量：

- `__launch_bounds__(N)`：直接指定核函数的**最大线程数** `N`，从而**影响**每个线程可用的寄存器数。不配置时默认为 1024 个线程。
- `__maxnreg__(N)`：直接指定**每个线程可用的寄存器数** `N`，从而**影响**核函数能启动的最大线程数。不配置时默认为 32 个寄存器。

> 注意：`__launch_bounds__(N)` 和 `__maxnreg__(N)` **不可同时配置**，二者只能选其一。（参照上表，默认的 1024 个线程与 32 个寄存器正好相互对应。）

使用示例如下：

```c++
__global__ __launch_bounds__(2048) void vec_add(float* x, float* y, float* z, int vector_length)
```

无论用哪个，目标都是一样的：**让最大线程数与算子复杂度相匹配**——计算复杂、局部变量多的算子，适当减少线程数以保证每个线程有足够寄存器，推荐配置最大线程数为512或1024；计算简单的算子，则可以增大线程数提高并行度，推荐配置最大线程数为 2048。具体SIMT算子需要多少寄存器，可通过编译器提供的编译选项`--cce-res-usage`查看，详细内容可参考文档[《合理配置最大线程数》](https://gitcode.com/cann/asc-devkit/blob/master/docs/guide/算子实践参考/SIMT算子性能优化/执行配置/合理配置线程数避免寄存器溢出.md)

### 术语速查

| 术语 | 说明 |
| --- | --- |
| Global Memory | 位于 Device 侧的全局内存，整个 Grid 内的所有线程均可访问；容量最大，且具有持久性。 |
| 共享内存 | 位于 Unified Buffer 内、供同一线程块内线程共享的内存；相较全局内存带宽更高、访问延迟更低。 |
| Unified Buffer | Vector Core 内部的片上存储资源，总大小为 256KB，划分为静态内存、动态内存、预留空间和 Data Cache。 |
| Data Cache | Unified Buffer 中用于缓存全局内存数据的区域，容量可配置范围为 32KB 至 128KB。 |
| Register（寄存器） | 每个线程私有的高速局部存储资源，在三类存储介质中访问速度最快。 |
| `__ubuf__` | 用于声明 Unified Buffer 内存的类型限定符。 |
| `__launch_bounds__(N)` | 可选配置，用于指定核函数的最大线程数，从而影响每个线程可用的寄存器数；默认为 1024 个线程。 |
| `__maxnreg__(N)` | 可选配置，用于指定每个线程可用的寄存器数，从而影响核函数的最大线程数；默认为 32 个寄存器，且与 `__launch_bounds__(N)` 不可同时配置。 |

## 小节小结

本小节从内存视角梳理了 SIMT 程序的三类存储介质及其配置约束：

- **三类存储介质**：Global Memory 是全局内存，所有线程可访问、容量最大、有持久性；共享内存位于 Vector Core 的 Unified Buffer，供线程块内线程共享、带宽高延迟低；Register 是线程私有的最快存储。规律是「作用域越小、越靠近计算单元、越快但越小」。
- **Unified Buffer 空间划分**：总大小 256KB，分为静态内存、动态内存、固定 8KB 预留空间和 Data Cache。Data Cache 由公式 `256KB − 静态 − 动态 − 8KB` 决定；其大小必须不低于 32KB（否则校验报错），且系统自动封顶为 128KB（即使算出更大也不会超过）。
- **寄存器与最大线程数**：最大线程数与每个线程可用寄存器数相互制约（最大线程数 256→127，2048→16）。线程数过大、计算复杂时容易触发 register spill。可用 `__launch_bounds__(N)` 或 `__maxnreg__(N)` 调整（二者只能选其一），关键是让最大线程数与算子复杂度相匹配。

写 SIMT 算子时，这两类配置——Unified Buffer 的空间划分、最大线程数与寄存器的平衡——是最容易影响性能、也最容易踩坑的地方。共享内存让线程块内的线程能够协作，但多个线程同时读写同一份数据时，如何保证顺序正确？这正是下一章节《同步机制》要解决的问题。

## 课后练习

本节介绍了 SIMT 的三类存储介质、Unified Buffer 的空间划分以及最大线程数与寄存器的关系，请根据学习内容完成以下题目进行自测。

1. （判断题）核函数定义时配置的最大线程数越大，每个线程可用的寄存器数量就越多。

2. （单选题）下列三类存储介质，按访问速度从快到慢排列，正确的是？  
    A. 全局内存 > 共享内存 > 寄存器  
    B. 寄存器 > 共享内存 > 全局内存  
    C. 共享内存 > 寄存器 > 全局内存  
    D. 寄存器 > 全局内存 > 共享内存  

3. （单选题）关于 Data Cache 的容量约束，下列说法正确的是？  
    A. 固定为 256KB  
    B. 可配置范围为 32KB ~ 128KB，不足 32KB 会校验报错  
    C. 最小 8KB，最大 256KB  
    D. 没有限制，由用户任意配置  

4. （多选题）以下关于寄存器与最大线程数的说法，哪些是正确的？  
    A. 寄存器是每个线程私有的存储资源  
    B. 发生寄存器溢出（register spill）会把数据放入更快的存储，从而提升性能  
    C. Unified Buffer 总大小 256KB，需为 Data Cache 至少保留 32KB  
    D. `__launch_bounds__(N)` 和 `__maxnreg__(N)` 都可用于调整线程数与寄存器的关系，但二者不可同时配置  

**执行以下代码获取答案。**

## 参考答案

运行下面的单元格查看课后练习参考答案。


In [ ]:
!cat answer/03.04.04_answer.txt
